In [4]:
import platform
import torch

print("=== Python ===")
print(platform.python_version())

print("\n=== PyTorch ===")
print(torch.__version__)

print("\n=== CUDA ===")
print("Available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)

    print("\n=== GPU ===")
    print("Name:", p.name)
    print(
        "Compute Capability:",
        f"{p.major}.{p.minor}",
    )
    print(
        "SM count:",
        p.multi_processor_count,
    )
    print(
        "VRAM:",
        f"{p.total_memory / 1024**3:.2f} GiB",
    )

    free, total = torch.cuda.mem_get_info()

    print(
        "Free VRAM:",
        f"{free / 1024**3:.2f} GiB",
    )

=== Python ===
3.13.16

=== PyTorch ===
2.11.0+cu130

=== CUDA ===
Available: True
PyTorch CUDA: 13.0

=== GPU ===
Name: Tesla T4
Compute Capability: 7.5
SM count: 40
VRAM: 14.56 GiB
Free VRAM: 14.46 GiB


In [5]:
!nvidia-smi

Wed Oct 28 03:20:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             25W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Capability:",
        torch.cuda.get_device_capability(0),
    )
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GiB",
    )

CUDA: True
GPU: Tesla T4
Capability: (7, 5)
VRAM: 14.56 GiB


In [4]:
import time
import torch

def cpu_matmul(n, repeat=20):
    a = torch.randn(n, n)
    b = torch.randn(n, n)

    # warmup
    _ = a @ b

    start = time.perf_counter()

    for _ in range(repeat):
        _ = a @ b

    end = time.perf_counter()

    return (end - start) / repeat


def gpu_matmul(n, repeat=20):
    device = "cuda"

    a = torch.randn(
        n,
        n,
        device=device,
    )

    b = torch.randn(
        n,
        n,
        device=device,
    )

    # GPU warmup
    for _ in range(5):
        _ = a @ b

    torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repeat):
        _ = a @ b

    torch.cuda.synchronize()

    end = time.perf_counter()

    return (end - start) / repeat


for n in [32, 128, 512, 2048]:
    cpu_t = cpu_matmul(n)

    gpu_t = gpu_matmul(n)

    print(
        f"N={n:4d} | "
        f"CPU={cpu_t*1000:9.3f} ms | "
        f"GPU={gpu_t*1000:9.3f} ms | "
        f"speedup={cpu_t/gpu_t:7.2f}x"
    )

N=  32 | CPU=    0.011 ms | GPU=    0.018 ms | speedup=   0.63x
N= 128 | CPU=    0.038 ms | GPU=    0.019 ms | speedup=   1.98x
N= 512 | CPU=    2.227 ms | GPU=    0.106 ms | speedup=  21.08x
N=2048 | CPU=  131.961 ms | GPU=    6.033 ms | speedup=  21.87x


In [12]:
import time
import torch

x = torch.randn(
    4096,
    4096,
)

torch.cuda.synchronize()

start = time.perf_counter()

y = x.to("cuda")

torch.cuda.synchronize()

end = time.perf_counter()

print(
    "CPU -> GPU:",
    (end - start) * 1000,
    "ms",
)


start = time.perf_counter()

z = y.to("cpu")

torch.cuda.synchronize()

end = time.perf_counter()

print(
    "GPU -> CPU:",
    (end - start) * 1000,
    "ms",
)

CPU -> GPU: 15.635927999937849 ms
GPU -> CPU: 47.9300159995546 ms


In [1]:
import torch

assert torch.cuda.is_available(), "请先把 Colab Runtime 切到 GPU"

props = torch.cuda.get_device_properties(0)

print("GPU:", props.name)
print("Compute Capability:", f"{props.major}.{props.minor}")
print("SM count:", props.multi_processor_count)
print("VRAM:", f"{props.total_memory / 1024**3:.2f} GiB")

for attr in [
    "max_threads_per_multi_processor",
    "warp_size",
]:
    if hasattr(props, attr):
        print(attr, "=", getattr(props, attr))

GPU: Tesla T4
Compute Capability: 7.5
SM count: 40
VRAM: 14.56 GiB
max_threads_per_multi_processor = 1024
warp_size = 32


In [3]:
print("PyTorch:", torch.__version__)
print("PyTorch CUDA Runtime:", torch.version.cuda)

PyTorch: 2.11.0+cu130
PyTorch CUDA Runtime: 13.0
